# 00 - Test rapide sur une phrase

Ce notebook permet de valider le comportement du moteur sans PDF. Saisis une phrase, ajuste les parametres si besoin, puis lance l'analyse avec le meme code que le traitement complet.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import importlib
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REFERENTIEL_DIR = PROJECT_ROOT / "input" / "referentiel"
CONFIG_FILE = PROJECT_ROOT / "config" / "scoring.yaml"

print("Projet :", PROJECT_ROOT)

## 1. Saisir le texte a tester

In [ ]:
texte_a_tester = """
Le client indique etre diabetique et souhaite adapter son contrat.
"""

texte_a_tester = texte_a_tester.strip()
print(texte_a_tester)

## 2. Charger les referentiels et parametrer la detection

In [ ]:
import load_referential
import detection

# Recharge les modules si le kernel Jupyter avait garde une ancienne version en memoire.
importlib.reload(load_referential)
importlib.reload(detection)

from load_referential import load_referentials
from detection import analyze_document

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Parametres modifiables pour le test
config["detection"]["activer_lemmatisation"] = True
config["detection"]["activer_synonymes_automatiques"] = True
config["detection"]["activer_mots_proches"] = True
config["detection"]["seuil_synonyme_automatique"] = 0.72
config["detection"]["seuil_similarite_orthographique"] = 88
config["score"]["synonyme_automatique"] = 80
config["detection"]["taille_contexte_mots"] = 12

referentials = load_referentials(REFERENTIEL_DIR)

display(referentials["mots_interdits"])
display(referentials["whitelist"])
config

## 3. Lancer l'analyse

In [ ]:
document_test = SimpleNamespace(name="phrase_saisie")
text_by_page = [{"page": 1, "text": texte_a_tester}]

results = analyze_document(document_test, text_by_page, referentials, config)
df_results = pd.DataFrame(results)

colonnes_resultat = [
    "document", "score", "score_unitaire", "statut", "terme_detecte",
    "terme_reference", "type_detection", "categorie", "motif", "page", "contexte"
]
for colonne in colonnes_resultat:
    if colonne not in df_results.columns:
        df_results[colonne] = ""

display(df_results[colonnes_resultat])

## 4. Vue simplifiee des alertes

In [ ]:
alertes = df_results[pd.to_numeric(df_results["score"], errors="coerce").fillna(0) > 0].copy()

if alertes.empty:
    print("Aucune alerte detectee.")
else:
    colonnes = ["score", "score_unitaire", "statut", "type_detection", "terme_detecte", "terme_reference", "categorie", "motif"]
    display(alertes[colonnes].sort_values(["score_unitaire", "type_detection"], ascending=[False, True]))

## 5. Detection zero-shot avec GLiNER

Cette cellule utilise un modele local telecharge depuis Hugging Face au premier lancement. Ensuite le modele peut etre reutilise depuis le cache local.

In [ ]:
# A executer une seule fois si GLiNER n'est pas installe dans ton environnement Python :
# %pip install gliner

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
    from gliner import GLiNER
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "GLiNER n'est pas installe dans le kernel Jupyter courant. "
        "Lance d'abord cette cellule dans le notebook : %pip install gliner"
    ) from exc

GLINER_MODEL = "urchade/gliner_multi-v2.1"
gliner_model = GLiNER.from_pretrained(GLINER_MODEL)

labels_rgpd = [
    "donnee de sante",
    "religion",
    "opinion politique",
    "appartenance syndicale",
    "origine ethnique",
    "vie sexuelle",
    "orientation sexuelle",
]

seuil_gliner = 0.35
entites_gliner = gliner_model.predict_entities(texte_a_tester, labels_rgpd, threshold=seuil_gliner)

df_gliner = pd.DataFrame(entites_gliner)
if df_gliner.empty:
    print("Aucune entite sensible detectee par GLiNER.")
else:
    colonnes_gliner = [col for col in ["text", "label", "score", "start", "end"] if col in df_gliner.columns]
    display(df_gliner[colonnes_gliner].sort_values("score", ascending=False))

## 6. Conversion GLiNER au format alertes

In [ ]:
mapping_categories = {
    "donnee de sante": "sante",
    "religion": "religion",
    "opinion politique": "opinion_politique",
    "appartenance syndicale": "appartenance_syndicale",
    "origine ethnique": "origine_ethnique",
    "vie sexuelle": "vie_sexuelle",
    "orientation sexuelle": "vie_sexuelle",
}

alertes_gliner = []
for entite in entites_gliner:
    label = entite.get("label", "")
    score_modele = float(entite.get("score", 0))
    alertes_gliner.append({
        "document": "phrase_saisie",
        "score": 80,
        "score_unitaire": 80,
        "statut": "Risque fort",
        "terme_detecte": entite.get("text", ""),
        "terme_reference": label,
        "type_detection": "gliner zero-shot",
        "categorie": mapping_categories.get(label, label),
        "motif": f"Entite detectee par GLiNER avec le label '{label}' et un score modele de {score_modele:.2f}",
        "page": 1,
        "contexte": texte_a_tester,
    })

df_alertes_gliner = pd.DataFrame(alertes_gliner)
if df_alertes_gliner.empty:
    print("Aucune alerte GLiNER a convertir.")
else:
    display(df_alertes_gliner[colonnes_resultat])